The Price is Right 

In [ ]:
import os
import json
import logging
from typing import List, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field

load_dotenv(override=True)


class Deal(BaseModel):
    product_description: str
    price: float
    url: str


class DealSelection(BaseModel):
    deals: List[Deal]


class Opportunity(BaseModel):
    deal: Deal
    estimate: float
    discount: float


class Agent:
    RED, GREEN, YELLOW, BLUE, MAGENTA, CYAN, WHITE = "\033[31m", "\033[32m", "\033[33m", "\033[34m", "\033[35m", "\033[36m", "\033[37m"
    BG_BLACK, RESET = "\033[40m", "\033[0m"
    name = ""
    color = "\033[37m"

    def log(self, message: str):
        logging.info(self.BG_BLACK + self.color + f"[{self.name}] {message}" + self.RESET)


logging.basicConfig(level=logging.INFO, format="[%(asctime)s] [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [ ]:

TEST_DEALS = [
    {"product_description": "The Hisense R6 Series 55R6030N is a 55-inch 4K UHD Roku Smart TV that offers stunning picture quality with 3840x2160 resolution. It features Dolby Vision HDR and HDR10 compatibility, ensuring a vibrant and dynamic viewing experience. The TV runs on Roku's operating system, allowing easy access to streaming services and voice control compatibility with Google Assistant and Alexa. With three HDMI ports available, connecting multiple devices is simple and efficient.", "price": 178.0, "url": "https://www.dealnews.com/products/Hisense/Hisense-R6-Series-55-R6030-N-55-4-K-UHD-Roku-Smart-TV/484824.html?iref=rss-c142"},
    {"product_description": "The Poly Studio P21 is a 21.5-inch LED personal meeting display designed specifically for remote work and video conferencing. With a native resolution of 1080p, it provides crystal-clear video quality, featuring a privacy shutter and stereo speakers. This display includes a 1080p webcam with manual pan, tilt, and zoom control, along with an ambient light sensor to adjust the vanity lighting as needed. It also supports 5W wireless charging for mobile devices, making it an all-in-one solution for home offices.", "price": 30.0, "url": "https://www.dealnews.com/products/Poly-Studio-P21-21-5-1080-p-LED-Personal-Meeting-Display/378335.html?iref=rss-c39"},
    {"product_description": "The Lenovo IdeaPad Slim 5 laptop is powered by a 7th generation AMD Ryzen 5 8645HS 6-core CPU, offering efficient performance for multitasking and demanding applications. It features a 16-inch touch display with a resolution of 1920x1080, ensuring bright and vivid visuals. Accompanied by 16GB of RAM and a 512GB SSD, the laptop provides ample speed and storage for all your files. This model is designed to handle everyday tasks with ease while delivering an enjoyable user experience.", "price": 446.0, "url": "https://www.dealnews.com/products/Lenovo/Lenovo-Idea-Pad-Slim-5-7-th-Gen-Ryzen-5-16-Touch-Laptop/485068.html?iref=rss-c39"},
    {"product_description": "The Dell G15 gaming laptop is equipped with a 6th-generation AMD Ryzen 5 7640HS 6-Core CPU, providing powerful performance for gaming and content creation. It features a 15.6-inch 1080p display with a 120Hz refresh rate, allowing for smooth and responsive gameplay. With 16GB of RAM and a substantial 1TB NVMe M.2 SSD, this laptop ensures speedy performance and plenty of storage for games and applications. Additionally, it includes the Nvidia GeForce RTX 3050 GPU for enhanced graphics and gaming experiences.", "price": 650.0, "url": "https://www.dealnews.com/products/Dell/Dell-G15-Ryzen-5-15-6-Gaming-Laptop-w-Nvidia-RTX-3050/485067.html?iref=rss-c39"},
]


class LiteScannerAgent(Agent):
    name = "Lite Scanner Agent"
    color = Agent.CYAN

    def __init__(self):
        self.log("Lite Scanner ready (inline test data)")

    def scan(self, memory: List[Opportunity]) -> Optional[DealSelection]:
        self.log("Returning test DealSelection (%d deals)" % len(TEST_DEALS))
        return DealSelection(deals=[Deal(**d) for d in TEST_DEALS])


scanner = LiteScannerAgent()

In [ ]:
import re
from openai import OpenAI


class EstimatorAgent(Agent):

    name = "Estimator Agent"
    color = Agent.YELLOW

    def __init__(self, model: str = "openai/gpt-4o-mini"):
        self.model = model
        key = os.getenv("OPEN_ROUTER_API_KEY")
        if not key:
            self.client = None
            self.log("No OPEN_ROUTER_API_KEY; estimates will return 0.0")
        else:
            self.client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=key)
        self.log("OpenRouter Estimator ready (model=%s)" % self.model)

    @staticmethod
    def _parse_price(text: str) -> float:
        text = text.replace("$", "").replace(",", "")
        m = re.search(r"[-+]?\d*\.?\d+", text)
        return float(m.group()) if m else 0.0

    def price(self, description: str) -> float:
        if not self.client:
            self.log("Skipping estimate (no API key); returning 0.0")
            return 0.0
        prompt = (
            "Estimate the typical retail price in USD for this product. "
            "Reply with only a number (e.g. 299.99), no explanation.\n\n"
            + description.strip()[:2000]
        )
        self.log("Calling OpenRouter for price estimate")
        try:
            r = self.client.chat.completions.create(
                model=self.model,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=32,
            )
            reply = (r.choices[0].message.content or "").strip()
            value = self._parse_price(reply)
            self.log("OpenRouter Estimator completed — $%.2f" % value)
            return value
        except Exception as e:
            self.log("OpenRouter error: %s; returning 0.0" % e)
            return 0.0


estimator = EstimatorAgent()

In [ ]:
import requests

PUSHOVER_URL = "https://api.pushover.net/1/messages.json"


class ConsoleMessagingAgent(Agent):
    name = "Console Messaging Agent"
    color = Agent.WHITE
    
    def __init__(self):
        self.pushover_user = os.getenv("PUSHOVER_USER", "")
        self.pushover_token = os.getenv("PUSHOVER_TOKEN", "")
        self.log("Console Messaging ready (push=%s)" % bool(self.pushover_token))
    def alert(self, opportunity: Opportunity):
        msg = (
            f"Deal alert — Price=${opportunity.deal.price:.2f}, "
            f"Estimate=${opportunity.estimate:.2f}, Discount=${opportunity.discount:.2f} | "
            + opportunity.deal.product_description[:80] + "... " + opportunity.deal.url
        )
        self.log("NOTIFY: %s" % msg)
        if self.pushover_token and self.pushover_user:
            try:
                requests.post(
                    PUSHOVER_URL,
                    data={
                        "user": self.pushover_user,
                        "token": self.pushover_token,
                        "message": msg[:1024],
                        "sound": "cashregister",
                    },
                    timeout=5,
                )
            except Exception as e:
                self.log("Pushover failed: %s" % e)


messenger = ConsoleMessagingAgent()

In [ ]:
class PlanningAgent(Agent):

    name = "Planning Agent"
    color = Agent.GREEN
    DEAL_THRESHOLD = 50

    def __init__(self, scanner, estimator, messenger):
        self.scanner = scanner
        self.ensemble = estimator 
        self.messenger = messenger
        self.log("Planning Agent ready")

    def run(self, deal: Deal) -> Opportunity:
        """Estimate one deal and return an Opportunity."""
        self.log("Pricing one deal")
        estimate = self.ensemble.price(deal.product_description)
        discount = estimate - deal.price
        self.log("Discount $%.2f" % discount)
        return Opportunity(deal=deal, estimate=estimate, discount=discount)

    def plan(self, memory: List[Opportunity]) -> Optional[Opportunity]:
        self.log("Starting plan run")
        selection = self.scanner.scan(memory=memory)
        if not selection or not selection.deals:
            self.log("No deals from scanner; done")
            return None
        opportunities = [self.run(d) for d in selection.deals[:5]]
        opportunities.sort(key=lambda o: o.discount, reverse=True)
        best = opportunities[0]
        self.log("Best discount $%.2f" % best.discount)
        if best.discount > self.DEAL_THRESHOLD:
            self.messenger.alert(best)
        self.log("Plan run complete")
        return best if best.discount > self.DEAL_THRESHOLD else None


planning_agent = PlanningAgent(scanner, estimator, messenger)

In [ ]:
PRODUCTS_MEMORY_FILE = "product_memory.json"


class DealFramework:
    def __init__(self, memory_path: str = PRODUCTS_MEMORY_FILE):
        self.memory_path = memory_path
        self.memory = self._read_memory()
        self.planner = None
        logger.info("DealFramework created; memory size = %d", len(self.memory))

    def _read_memory(self) -> List[Opportunity]:
        if os.path.exists(self.memory_path):
            with open(self.memory_path, "r") as f:
                data = json.load(f)
            return [Opportunity(**item) for item in data]
        return []

    def _write_memory(self):
        data = [o.model_dump() for o in self.memory]
        with open(self.memory_path, "w") as f:
            json.dump(data, f, indent=2)

    def init_agents(self):
        if self.planner is None:
            logger.info("Initializing agents")
            self.planner = ExercisePlanningAgent(scanner, estimator, messenger)
            logger.info("Agents ready")

    def run(self) -> List[Opportunity]:
        self.init_agents()
        result = self.planner.plan(memory=self.memory)
        if result:
            self.memory.append(result)
            self._write_memory()
        return self.memory


framework = DealFramework()

In [ ]:
opportunities = framework.run()
for i, opp in enumerate(opportunities):
    print(f"{i+1}. {opp.deal.product_description[:60]}... | Price ${opp.deal.price:.2f} | Est ${opp.estimate:.2f} | Disc ${opp.discount:.2f}")

In [ ]:
import queue
import threading
import time
import numpy as np
import gradio as gr
import plotly.graph_objects as go

BG_BLACK, RED, GREEN, YELLOW, BLUE, MAGENTA, CYAN, WHITE, BG_BLUE, RESET = (
    "\033[40m", "\033[31m", "\033[32m", "\033[33m", "\033[34m", "\033[35m", "\033[36m", "\033[37m", "\033[44m", "\033[0m"
)
COLOR_MAP = {
    BG_BLACK + RED: "#dd0000", BG_BLACK + GREEN: "#00dd00", BG_BLACK + YELLOW: "#dddd00",
    BG_BLACK + BLUE: "#0000ee", BG_BLACK + MAGENTA: "#aa00dd", BG_BLACK + CYAN: "#00dddd",
    BG_BLACK + WHITE: "#87CEEB", BG_BLUE + WHITE: "#ff7800",
}


def reformat_log(message: str) -> str:
    for k, v in COLOR_MAP.items():
        message = message.replace(k, f'<span style="color: {v}">')
    return message.replace(RESET, "</span>")


class QueueHandler(logging.Handler):
    def __init__(self, log_queue):
        super().__init__()
        self.log_queue = log_queue

    def emit(self, record):
        self.log_queue.put(self.format(record))


def setup_logging(log_queue):
    h = QueueHandler(log_queue)
    h.setFormatter(logging.Formatter("[%(asctime)s] %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
    root = logging.getLogger()
    root.addHandler(h)
    root.setLevel(logging.INFO)


def html_for(log_data):
    out = "<br>".join(log_data[-20:])
    return f"""<div style="height: 420px; overflow-y: auto; border: 1px solid #444; background-color: #1a1a1a; padding: 12px; font-family: monospace; font-size: 13px; color: #fff;">{out}</div>"""


def get_plot():
    n = 100
    vecs = np.random.randn(n, 3)
    colors = ["red", "blue", "green", "orange"] * (n // 4 + 1)
    fig = go.Figure(data=[go.Scatter3d(x=vecs[:, 0], y=vecs[:, 1], z=vecs[:, 2], mode="markers", marker=dict(size=3, color=colors[:n], opacity=0.7))])
    fig.update_layout(
        scene=dict(xaxis_title="X", yaxis_title="Y", zaxis_title="Z", aspectmode="manual", aspectratio=dict(x=2.2, y=2.2, z=1), camera=dict(eye=dict(x=1.6, y=1.6, z=0.8)), bgcolor="#1a1a1a"),
        height=420, margin=dict(r=5, b=5, l=5, t=5), paper_bgcolor="#1a1a1a", font=dict(color="#ffffff"), title="Vector space (demo)",
    )
    return fig


def opportunities_to_table(opps: List[Opportunity]):
    if not opps:
        return []
    return [[o.deal.product_description, f"${o.deal.price:.2f}", f"${o.estimate:.2f}", f"${o.discount:.2f}", o.deal.url] for o in opps]


def scan_with_logging(log_data):
    log_queue = queue.Queue()
    result_queue = queue.Queue()
    setup_logging(log_queue)

    def worker():
        framework.run()
        result_queue.put(opportunities_to_table(framework.memory))

    threading.Thread(target=worker).start()
    current = opportunities_to_table(framework.memory)

    while True:
        try:
            msg = log_queue.get_nowait()
            log_data.append(reformat_log(msg))
            current = opportunities_to_table(framework.memory)
            yield log_data, html_for(log_data), current
        except queue.Empty:
            try:
                final = result_queue.get_nowait()
                yield log_data, html_for(log_data), final
                return
            except queue.Empty:
                current = opportunities_to_table(framework.memory)
                yield log_data, html_for(log_data), current
                time.sleep(0.1)


def load_initial_state():
    return [], html_for([]), opportunities_to_table(framework.memory)


def handle_selection(selected_index: gr.SelectData):
    row = selected_index.index[0]
    if row < len(framework.memory):
        framework.planner.messenger.alert(framework.memory[row])
        return f"Alert sent for: {framework.memory[row].deal.product_description[:60]}..."
    return "Invalid selection"


TIMER_SECONDS = 60

framework.init_agents()

with gr.Blocks(title="The Price is Right", fill_width=True) as ui:
    log_data = gr.State([])
    gr.Markdown(
        "<div style='text-align: center; font-size: 28px; font-weight: bold; margin: 20px 0;'>The Price is Right - Week 8 Exercise</div>"
        + "<div style='text-align: center; font-size: 16px; color: #666; margin-bottom: 20px;'>Table above, logs + plot below. Agent runs on timer only (every " + str(TIMER_SECONDS) + " s).</div>"
    )
    with gr.Row():
        opportunities_table = gr.Dataframe(
            headers=["Product Description", "Price", "Estimate", "Discount", "URL"],
            wrap=True, column_widths=[6, 1, 1, 1, 3], row_count=10, col_count=5, max_height=420, interactive=False,
        )
    with gr.Row():
        with gr.Column(scale=1):
            logs_display = gr.HTML(label="Agent Logs")
        with gr.Column(scale=1):
            vector_plot = gr.Plot(value=get_plot(), show_label=False)
    ui.load(load_initial_state, inputs=[], outputs=[log_data, logs_display, opportunities_table])
    timer = gr.Timer(value=TIMER_SECONDS, active=True)
    timer.tick(scan_with_logging, inputs=[log_data], outputs=[log_data, logs_display, opportunities_table])
    selection_feedback = gr.Textbox(visible=False)
    opportunities_table.select(handle_selection, inputs=[], outputs=[selection_feedback])
ui.launch(share=False, inbrowser=True)